# Phase 1b: From a Raw Phone Video to a Gaussian Splat

Garden was the easy case: clean photos, plus official poses to check against. Here we start from **your own video**, with no reference to lean on.

**Pipeline:** video → FFmpeg frames → keep the sharpest frame in each time window → COLMAP (sequential matching + loop detection) → pose QA → undistort → gsplat (7k steps) → metrics + `.ply`

**Before you run this**
1. Film your scene using `docs/capture_guide.md`.
2. Upload it to Google Drive as `MyDrive/3dgs-digital-twin/videos/scene1.mp4`. Other names work if you change `SCENE` below; `.mov` works too.
3. `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. If the runtime restarts itself once, press `Run all` again.

**Not in this notebook:** masking with SAM 2. If you film a still scene, it isn't needed yet. It arrives with the Wayve driving scene, where moving cars make it necessary.

⏱️ Rough total on a T4: 30-50 min the first time, including a one-time compile of gsplat. Later sessions reuse the compiled package and the cached SfM result.

## 1. GPU, Drive, settings

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, json, time, re, subprocess, shutil
SCENE = 'scene1'            # video file name without extension

# ---- knobs ----
TARGET_FRAMES = 300         # frames kept for SfM + training
OVERSAMPLE    = 3           # extract 3x as many, keep the sharpest in each window
MAX_SIDE      = 1600        # longest image side in pixels (same limit as the official 3DGS convert.py)
MATCHER       = 'sequential'  # switch to 'exhaustive' if registration is poor (slower, more robust)
STEPS         = 7000

PROJECT = '/content/drive/MyDrive/3dgs-digital-twin'
WORK    = '/content/work'
OUT     = f'{PROJECT}/results/phase1b_{SCENE}_sfm'
for d in [f'{PROJECT}/results', f'{PROJECT}/videos', f'{PROJECT}/exports', f'{PROJECT}/cache/torch_extensions', WORK, OUT]:
    os.makedirs(d, exist_ok=True)

## 2. Install gsplat + COLMAP

The same install cells as the garden notebook, with every fix from `docs/failures.md` built in.

In [ ]:
import importlib.util
MARKER = '/content/.gs_deps_ok'          # /content survives a kernel restart (not a VM reset)

if not os.path.exists(MARKER):
    import torch
    np_before = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                               capture_output=True, text=True).stdout.strip()
    # gsplat: prebuilt wheel if available, else PyPI (JIT compile)
    tv = torch.__version__.split('+')[0].split('.'); cu = (torch.version.cuda or '').replace('.', '')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat==1.5.3',
                        '--index-url', f'https://docs.gsplat.studio/whl/pt{tv[0]}{tv[1]}cu{cu}',
                        '--extra-index-url', 'https://pypi.org/simple'], capture_output=True, text=True)
    if r.returncode != 0:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat==1.5.3'], check=True)

    REPO = f'{WORK}/gsplat'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.5.3',
                        'https://github.com/nerfstudio-project/gsplat.git', REPO], check=True)
    reqs = open(f'{REPO}/examples/requirements.txt').read()

    # F0.1 + F0.2: git deps (several glued onto one line) -> split, build without isolation
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'setuptools', 'wheel'], check=True)
    GPU_NAME = torch.cuda.get_device_name(0)
    if 'T4' in GPU_NAME:
        os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'   # faster builds, T4 only
    for url in re.findall(r'git\+https://github\.com/[^@\s]+@[0-9a-f]{40}', reqs):
        print('building', url.split('/')[-1][:60])
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', url], check=True)

    # F0.3: drop git lines + numpy<2 pin, install the rest
    rest = [l for l in reqs.splitlines() if 'git+https' not in l and not l.lower().startswith('numpy')]
    open('/tmp/reqs.txt', 'w').write('\n'.join(rest))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/tmp/reqs.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'remotezip', 'plyfile', 'numpy>=2.1,<2.5'], check=True)

    # F0.5: pycolmap fork uses np.uint64(-1) -> OverflowError on numpy 2 (patch the file, don't import it)
    f = os.path.join(os.path.dirname(importlib.util.find_spec('pycolmap').origin), 'scene_manager.py')
    s = open(f).read()
    if 'np.uint64(-1)' in s:
        open(f, 'w').write(s.replace('np.uint64(-1)', 'np.uint64(18446744073709551615)')); print('patched pycolmap')

    # F1.5: otherwise gsplat recompiles its CUDA code at the first render of every session (~13 min).
    # Build a compiled package once, keep it on Drive, and install that in later sessions.
    if not subprocess.run([sys.executable, '-c', 'from gsplat import csrc'], capture_output=True).returncode == 0:
        WHEELS = (f"{PROJECT}/cache/wheels/gsplat153_pt{torch.__version__.split('+')[0]}_cu{cu}"
                  f"_py{sys.version_info.major}{sys.version_info.minor}_{GPU_NAME.replace(' ', '-')}")
        os.makedirs(WHEELS, exist_ok=True)
        if not glob.glob(f'{WHEELS}/gsplat-*.whl'):
            print('Compiling gsplat once for this GPU + torch (10-20 min). Later sessions reuse it.')
            r = subprocess.run([sys.executable, '-m', 'pip', 'wheel', '-q', '--no-deps', '--no-build-isolation', '-w', WHEELS,
                                'git+https://github.com/nerfstudio-project/gsplat.git@v1.5.3'],
                               capture_output=True, text=True)
            if r.returncode != 0:
                print('Building failed, gsplat will compile during training instead:', r.stderr[-800:])
        for whl in glob.glob(f'{WHEELS}/gsplat-*.whl'):
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', whl], check=True)
    print('gsplat precompiled:', subprocess.run([sys.executable, '-c', 'from gsplat import csrc'], capture_output=True).returncode == 0)

    # F0.4: make gsplat's examples/datasets a regular package (beats HF `datasets`)
    open(f'{REPO}/examples/datasets/__init__.py', 'a').close()

    open(MARKER, 'w').write('ok')
    np_after = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                              capture_output=True, text=True).stdout.strip()
    if np_before != np_after:
        print(f'numpy {np_before} -> {np_after}: restarting runtime. Then "Run all" again.')
        time.sleep(2); os.kill(os.getpid(), 9)

REPO = f'{WORK}/gsplat'
check = subprocess.run([sys.executable, '-c',
    'import numpy, gsplat, fused_ssim, pycolmap, tyro; print("numpy", numpy.__version__, "| gsplat", gsplat.__version__)'],
    capture_output=True, text=True, cwd=f'{REPO}/examples')
print(check.stdout or check.stderr[-2000:])

In [ ]:
MAMBA   = f'{WORK}/bin/micromamba'
SFM_ENV = f'{WORK}/sfm'
os.environ['MAMBA_ROOT_PREFIX'] = f'{WORK}/mamba'
if not os.path.exists(MAMBA):
    subprocess.run(f'curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C {WORK} bin/micromamba',
                   shell=True, check=True)

COLMAP = f'{SFM_ENV}/bin/colmap'
SFM_ENVVARS = {**os.environ, 'QT_QPA_PLATFORM': 'offscreen'}

def colmap_banner():
    r = subprocess.run([COLMAP, '-h'], capture_output=True, text=True, env=SFM_ENVVARS)
    return r.returncode, r.stdout + r.stderr

# Known-good combo (see docs/failures.md F1.1). Don't add the separate `glomap` package: it breaks this build.
if not os.path.exists(COLMAP) or colmap_banner()[0] != 0:
    shutil.rmtree(SFM_ENV, ignore_errors=True); t = time.time()
    subprocess.run([MAMBA, 'create', '-y', '-q', '-p', SFM_ENV, '-c', 'conda-forge',
                    'colmap=4.2.1=*cuda*', 'libfaiss=*=*cuda*'], check=True)
    print(f'installed in {time.time()-t:.0f}s')

rc, hdr = colmap_banner()
print('\n'.join(hdr.splitlines()[:2]))
if rc != 0:
    raise RuntimeError('COLMAP does not start, see the error above')   # never mistake this for "CPU-only"
COLMAP_CUDA = 'with CUDA' in hdr
# COLMAP 4 may include GLOMAP's global SfM as a built-in command
HAS_GLOBAL_MAPPER = subprocess.run([COLMAP, 'global_mapper', '-h'], capture_output=True, env=SFM_ENVVARS).returncode == 0
print('COLMAP CUDA:', COLMAP_CUDA, '| global_mapper:', HAS_GLOBAL_MAPPER)
if not COLMAP_CUDA:
    print('⚠️ COLMAP runs but without CUDA: SIFT + exhaustive matching will be slow (~30-60 min).')

In [ ]:
# helpers: run a command with logging, and pick option names that exist in this COLMAP version
SFM_LOG = f'{OUT}/sfm_log.txt'

def sh(cmd, tail=12, cwd=None):
    t = time.time()
    p = subprocess.run([str(c) for c in cmd], capture_output=True, text=True, env=SFM_ENVVARS, cwd=cwd)
    out = p.stdout + p.stderr
    with open(SFM_LOG, 'a') as f:
        f.write(f'\n$ {" ".join(map(str, cmd))}\n{out}\n')
    dt = time.time() - t
    lines = [l for l in out.strip().splitlines() if l.strip()]
    if p.returncode != 0:
        print('\n'.join(lines[-40:]))
        raise RuntimeError(f'{os.path.basename(str(cmd[0]))} {cmd[1]} failed (exit {p.returncode}), full log: {SFM_LOG}')
    if tail: print('\n'.join(lines[-tail:]))
    print(f'✔ {os.path.basename(str(cmd[0]))} {cmd[1]}: {dt/60:.1f} min')
    return out, dt

_help_cache = {}
def opt(sub, *candidates):
    # COLMAP renamed some options between versions (e.g. SiftExtraction.use_gpu -> FeatureExtraction.use_gpu)
    if sub not in _help_cache:
        r = subprocess.run([COLMAP, sub, '-h'], capture_output=True, text=True, env=SFM_ENVVARS)
        _help_cache[sub] = r.stdout + r.stderr
    for c in candidates:
        if f'--{c}' in _help_cache[sub]:
            return c
    return None
print('extract gpu opt:', opt('feature_extractor', 'FeatureExtraction.use_gpu', 'SiftExtraction.use_gpu'))
print('match gpu opt:  ', opt('exhaustive_matcher', 'FeatureMatching.use_gpu', 'SiftMatching.use_gpu'))

## 3. Check the video

In [ ]:
candidates = [f'{PROJECT}/videos/{SCENE}{ext}' for ext in ('.mp4', '.MP4', '.mov', '.MOV')]
VIDEO = next((v for v in candidates if os.path.exists(v)), None)
if VIDEO is None:
    raise FileNotFoundError(f'Upload your video to Drive as {candidates[0]} (or change SCENE in section 1)')
LOCAL_VIDEO = f'{WORK}/{os.path.basename(VIDEO)}'
if not os.path.exists(LOCAL_VIDEO):
    shutil.copy(VIDEO, LOCAL_VIDEO)

probe = json.loads(subprocess.run(
    ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-of', 'json', '-show_entries',
     'stream=width,height,codec_name,avg_frame_rate,color_transfer,pix_fmt:format=duration,size', LOCAL_VIDEO],
    capture_output=True, text=True, check=True).stdout)
st = probe['streams'][0]
duration = float(probe['format']['duration'])
num, den = st['avg_frame_rate'].split('/')
src_fps = float(num) / float(den)
print(f"{st['width']}x{st['height']} | {st['codec_name']} {st.get('pix_fmt')} | {src_fps:.1f} fps | {duration:.0f} s | "
      f"{int(probe['format']['size'])/1e6:.0f} MB | color: {st.get('color_transfer', '?')}")

if st.get('color_transfer') in ('smpte2084', 'arib-std-b67'):
    print('⚠️ HDR video: frames may look washed out. Re-film with HDR off if they do (section 4 shows samples).')
if max(st['width'], st['height']) < 1280:
    print('⚠️ Low resolution: is this the original phone file? Fine detail will be limited. Use the original if you have it.')
if duration < 20:
    print('⚠️ Very short video: probably too few viewpoints.')
if duration > 300:
    print('⚠️ Long video: frames will be far apart. Keep it around 1-3 minutes for a first run.')

## 4. Extract frames, keep the sharp ones

Dropping the blurriest frames *overall* can leave holes in coverage when a whole stretch of video is shaky. So we extract 3× as many frames and keep the **sharpest frame in each time window**, which spaces them evenly. Frames that are still very blurry after that are dropped too.

In [ ]:
import numpy as np, imageio.v2 as imageio, matplotlib.pyplot as plt

def blur_score(path):
    g = imageio.imread(path).astype(np.float32)
    g = g[..., :3].mean(-1) if g.ndim == 3 else g
    lap = g[1:-1, 2:] + g[1:-1, :-2] + g[2:, 1:-1] + g[:-2, 1:-1] - 4 * g[1:-1, 1:-1]
    return float(lap.var())

def pick_sharpest(scores, window, min_ratio=0.25):
    # the best frame per window, minus any that are far blurrier than the typical pick
    scores = np.asarray(scores, dtype=float)
    picks = [s + int(np.argmax(scores[s:s + window])) for s in range(0, len(scores), window)]
    floor = min_ratio * np.median(scores[picks])
    return [i for i in picks if scores[i] >= floor]

RAW    = f'{WORK}/{SCENE}/raw_frames'
IMAGES = f'{WORK}/{SCENE}/frames'
if not (os.path.isdir(IMAGES) and os.listdir(IMAGES)):
    shutil.rmtree(RAW, ignore_errors=True); shutil.rmtree(IMAGES, ignore_errors=True)
    os.makedirs(RAW); os.makedirs(IMAGES)
    fps = min(src_fps, TARGET_FRAMES * OVERSAMPLE / duration)
    side = min(MAX_SIDE, max(st['width'], st['height']))
    t = time.time()
    subprocess.run(['ffmpeg', '-y', '-v', 'error', '-i', LOCAL_VIDEO, '-an',
                    '-vf', f'fps={fps:.4f},scale={side}:{side}:force_original_aspect_ratio=decrease',
                    '-qscale:v', '2', f'{RAW}/%05d.jpg'], check=True)
    raw = sorted(os.listdir(RAW))
    print(f'extracted {len(raw)} frames at {fps:.2f} fps in {time.time()-t:.0f}s')
    scores = [blur_score(f'{RAW}/{n}') for n in raw]
    keep = pick_sharpest(scores, OVERSAMPLE)
    for i in keep:
        shutil.copy(f'{RAW}/{raw[i]}', f'{IMAGES}/{raw[i]}')
    print(f'kept {len(keep)} of {len(raw)} frames')
    plt.figure(figsize=(14, 2.5)); plt.plot(scores, lw=0.7, label='all frames')
    plt.plot(keep, np.array(scores)[keep], '.', ms=3, label='kept'); plt.xlabel('frame'); plt.ylabel('sharpness')
    plt.legend(); plt.title('Sharpness over the video'); plt.savefig(f'{OUT}/sharpness.png', dpi=100, bbox_inches='tight'); plt.show()
    shutil.rmtree(RAW)

files = sorted(os.listdir(IMAGES))
fig, ax = plt.subplots(1, 5, figsize=(18, 3.5))
for a, n in zip(ax, files[:: max(1, len(files) // 5)]):
    a.imshow(imageio.imread(f'{IMAGES}/{n}')); a.set_title(n, fontsize=8); a.axis('off')
plt.show()
print(len(files), 'frames | size', imageio.imread(f'{IMAGES}/{files[0]}').shape)

**Look at the samples above before continuing.** Are they sharp, right way up, and not washed out? Are people or cars moving through them? Fixing the video now saves an hour later.

## 5. Run COLMAP

**Sequential matching** compares each frame with its neighbours instead of every pair, which suits video. **Loop detection** also finds matches when you come back to a place you already filmed. Without it, small errors build up along the path and the two ends of a loop won't line up.

The result is cached on Drive. Delete `results/phase1b_<scene>_sfm/sparse_colmap` to force a re-run, for example after changing the video.

In [ ]:
import struct
SFM = f'{WORK}/{SCENE}/sfm'
DB  = f'{SFM}/database.db'
os.makedirs(SFM, exist_ok=True)
CACHE = f'{OUT}/sparse_colmap'
timings = json.load(open(f'{OUT}/timings.json')) if os.path.exists(f'{OUT}/timings.json') else {}

def n_images_in(model_dir):
    with open(f'{model_dir}/images.bin', 'rb') as f:
        return struct.unpack('<Q', f.read(8))[0]

other_models = []
if os.path.exists(f'{CACHE}/images.bin'):
    print('✔ using cached COLMAP model from Drive'); BEST = CACHE
else:
    open(SFM_LOG, 'w').close()
    if os.path.exists(DB): os.remove(DB)
    shutil.rmtree(f'{SFM}/sparse_raw', ignore_errors=True)
    G = '1' if COLMAP_CUDA else '0'

    cmd = [COLMAP, 'feature_extractor', '--database_path', DB, '--image_path', IMAGES,
           '--ImageReader.single_camera', '1', '--ImageReader.camera_model', 'OPENCV']
    gpu = opt('feature_extractor', 'FeatureExtraction.use_gpu', 'SiftExtraction.use_gpu')
    if gpu: cmd += [f'--{gpu}', G]
    _, timings['extract_s'] = sh(cmd)

    def match(kind, loop=True):
        sub = f'{kind}_matcher'
        cmd = [COLMAP, sub, '--database_path', DB]
        gpu = opt(sub, 'FeatureMatching.use_gpu', 'SiftMatching.use_gpu')
        if gpu: cmd += [f'--{gpu}', G]
        if kind == 'sequential':
            ov = opt(sub, 'SequentialMatching.overlap')
            if ov: cmd += [f'--{ov}', '20']
            ld = opt(sub, 'SequentialMatching.loop_detection')
            if ld and loop: cmd += [f'--{ld}', '1']
        return sh(cmd)[1]

    if MATCHER == 'sequential':
        try:
            timings['match_s'] = match('sequential')
        except RuntimeError as e:
            print('Loop detection failed, matching neighbours only:', e)
            timings['match_s'] = match('sequential', loop=False)
    else:
        timings['match_s'] = match('exhaustive')

    os.makedirs(f'{SFM}/sparse_raw')
    cmd = [COLMAP, 'mapper', '--database_path', DB, '--image_path', IMAGES, '--output_path', f'{SFM}/sparse_raw']
    tol = opt('mapper', 'Mapper.ba_global_function_tolerance')
    if tol: cmd += [f'--{tol}', '0.000001']
    _, timings['mapper_s'] = sh(cmd, tail=5)

    models = [os.path.dirname(m) for m in glob.glob(f'{SFM}/sparse_raw/*/images.bin')]
    if not models:
        raise RuntimeError('COLMAP built no model: the video probably has too little overlap or texture. See the capture guide.')
    print('models found (images in each):', sorted((n_images_in(m) for m in models), reverse=True))
    BEST = max(models, key=n_images_in)
    other_models = [n_images_in(m) for m in models if m != BEST]
    shutil.copytree(BEST, CACHE, dirs_exist_ok=True)
    json.dump(timings, open(f'{OUT}/timings.json', 'w'), indent=1)
print('timings (min):', {k: round(v / 60, 1) for k, v in timings.items()})
print('registered', n_images_in(BEST), 'of', len(os.listdir(IMAGES)), 'frames')

## 6. Pose QA without a reference

There are no official poses here, so we check three things:
1. **COLMAP's statistics:** how many frames were registered, and the reprojection error.
2. **The camera path:** you moved smoothly, so the path should be smooth too. A sudden jump suggests a bad pose. We also look for runs of frames that weren't registered.
3. **The point cloud:** does it look like your scene from above and from the side? This is the check that catches warped or doubled geometry, and only you can judge it.

⚠️ Passing these doesn't prove the poses are accurate the way the garden comparison did. They only catch visible failures.

In [ ]:
import struct
def read_images_bin(path):
    imgs = {}
    with open(path, 'rb') as f:
        n = struct.unpack('<Q', f.read(8))[0]
        for _ in range(n):
            struct.unpack('<i', f.read(4))                       # image_id
            q = np.array(struct.unpack('<4d', f.read(32)))        # qw qx qy qz (world -> camera)
            t = np.array(struct.unpack('<3d', f.read(24)))
            struct.unpack('<i', f.read(4))                       # camera_id
            name = b''
            while (c := f.read(1)) != b'\x00':
                name += c
            n2d = struct.unpack('<Q', f.read(8))[0]
            f.seek(24 * n2d, 1)                                   # skip 2D points (x, y, point3D_id)
            imgs[os.path.basename(name.decode())] = (q, t)
    return imgs

def qvec2rotmat(q):
    w, x, y, z = q / np.linalg.norm(q)
    return np.array([[1-2*y*y-2*z*z, 2*x*y-2*w*z,   2*x*z+2*w*y],
                     [2*x*y+2*w*z,   1-2*x*x-2*z*z, 2*y*z-2*w*x],
                     [2*x*z-2*w*y,   2*y*z+2*w*x,   1-2*x*x-2*y*y]])


def read_points3D_bin(path):
    xyz, rgb, err = [], [], []
    with open(path, 'rb') as f:
        for _ in range(struct.unpack('<Q', f.read(8))[0]):
            f.read(8)                                                   # point id
            xyz.append(struct.unpack('<3d', f.read(24)))
            rgb.append(struct.unpack('<3B', f.read(3)))
            err.append(struct.unpack('<d', f.read(8))[0])
            track = struct.unpack('<Q', f.read(8))[0]
            f.seek(8 * track, 1)                                        # skip the (image_id, point2D_idx) track
    return np.array(xyz), np.array(rgb) / 255.0, np.array(err)

def path_stats(centers, jump_ratio=5.0):
    # jumps = steps between consecutive registered frames that are far longer than the typical step
    steps = np.linalg.norm(np.diff(centers, axis=0), axis=1)
    median = float(np.median(steps))
    return steps, median, np.where(steps > jump_ratio * median)[0]

def longest_gap(all_names, registered):
    # the longest run of consecutive frames that COLMAP failed to register
    best = run = 0
    for n in all_names:
        run = 0 if n in registered else run + 1
        best = max(best, run)
    return best

In [ ]:
def analyze(model_dir, label):
    out, _ = sh([COLMAP, 'model_analyzer', '--path', model_dir], tail=0)
    def grab(key):
        m = re.search(key + r'[^:]*:\s*([0-9.]+)', out)
        return float(m.group(1)) if m else None
    st = {'model': label,
          'registered': n_images_in(model_dir),   # read from the model, not from log text
          'total_images': len(os.listdir(IMAGES)),
          'points3D': int.from_bytes(open(f'{model_dir}/points3D.bin', 'rb').read(8), 'little'),
          'mean_track_length': grab('Mean track length'),
          'mean_obs_per_image': grab('Mean observations per image'),
          'mean_reproj_error_px': grab('Mean reprojection error')}
    st['registered_pct'] = round(100 * st['registered'] / st['total_images'], 1)
    return st

stats = analyze(BEST, 'colmap')
imgs = read_images_bin(f'{BEST}/images.bin')
all_names = sorted(os.listdir(IMAGES))
names = [n for n in all_names if n in imgs]
Rs = [qvec2rotmat(imgs[n][0]) for n in names]
C = np.stack([-R.T @ imgs[n][1] for R, n in zip(Rs, names)])
steps, med, jumps = path_stats(C)
gap = longest_gap(all_names, set(imgs))
xyz, rgb, err = read_points3D_bin(f'{BEST}/points3D.bin')

checks = {
    'registered > 95%': stats['registered_pct'] > 95,
    'reprojection < 1 px': stats['mean_reproj_error_px'] is not None and stats['mean_reproj_error_px'] < 1,
    'no jumps in the camera path': len(jumps) == 0,
    'no gap of 10+ unregistered frames': gap < 10,
    'one main model (no split)': not other_models or max(other_models) < 0.1 * stats['registered'],
}
print(json.dumps(stats, indent=1))
print(f'path jumps: {len(jumps)} | longest unregistered gap: {gap} frames | 3D points: {len(xyz):,}')
for name, ok in checks.items():
    print(f"  {'✅' if ok else '❌'} {name}")
if jumps.size:
    print('  jumps after frames:', [names[i] for i in jumps[:10]])

# Top-down and side views, using the main axes of the point cloud
keep = np.all((xyz > np.percentile(xyz, 1, axis=0)) & (xyz < np.percentile(xyz, 99, axis=0)), axis=1)
center = xyz[keep].mean(0)
_, _, V = np.linalg.svd(xyz[keep] - center, full_matrices=False)
P, Pc = (xyz[keep] - center) @ V.T, (C - center) @ V.T
fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))
ax[0].scatter(P[:, 0], P[:, 1], c=rgb[keep], s=0.4); ax[0].plot(Pc[:, 0], Pc[:, 1], 'r-', lw=0.8, label='camera path')
ax[0].set_title('Point cloud + camera path (top-down)'); ax[0].set_aspect('equal'); ax[0].legend()
ax[1].scatter(P[:, 0], P[:, 2], c=rgb[keep], s=0.4); ax[1].plot(Pc[:, 0], Pc[:, 2], 'r-', lw=0.8)
ax[1].set_title('Side view'); ax[1].set_aspect('equal')
ax[2].plot(steps, lw=0.8); ax[2].axhline(5 * med, color='r', ls='--', label='jump threshold')
ax[2].set_title('Camera movement between frames'); ax[2].set_xlabel('frame'); ax[2].legend()
plt.tight_layout(); plt.savefig(f'{OUT}/pose_qa.png', dpi=110); plt.show()

json.dump({'stats': stats, 'path_jumps': len(jumps), 'longest_gap': gap, 'checks': {k: bool(v) for k, v in checks.items()},
           'timings_s': timings}, open(f'{OUT}/pose_qa.json', 'w'), indent=1)
if not all(checks.values()):
    print('\n⚠️ Some checks failed. Look at the plots before spending time on training. See "If COLMAP struggles" in the capture guide.')

## 7. Undistort → training-ready scene

In [ ]:
SCENE_DIR = f'{WORK}/data/{SCENE}'
if not os.path.exists(f'{SCENE_DIR}/sparse/0/images.bin'):
    shutil.rmtree(SCENE_DIR, ignore_errors=True)
    os.makedirs(SCENE_DIR, exist_ok=True)      # COLMAP aborts if the parent folder is missing
    sh([COLMAP, 'image_undistorter', '--image_path', IMAGES, '--input_path', BEST,
        '--output_path', SCENE_DIR, '--output_type', 'COLMAP'], tail=3)
    os.makedirs(f'{SCENE_DIR}/sparse/0', exist_ok=True)
    for fn in glob.glob(f'{SCENE_DIR}/sparse/*.bin'):
        shutil.move(fn, f'{SCENE_DIR}/sparse/0/')
print(len(os.listdir(f'{SCENE_DIR}/images')), 'images | size', imageio.imread(glob.glob(f'{SCENE_DIR}/images/*')[0]).shape)

## 8. Train (7k steps)

The images are already at training size, so we use `--data_factor 1`. Every 8th frame is held out for testing.

⚠️ Video frames are close together, so a held-out frame sits right next to training frames. That makes this test easier than garden's, and **these scores can't be compared with garden's**. Judge the result mostly by flying through it.

In [ ]:
help_txt = subprocess.run([sys.executable, 'simple_trainer.py', 'default', '--help'], capture_output=True,
                          text=True, cwd=f'{REPO}/examples').stdout
LPIPS = ['--lpips_net', 'vgg'] if re.search(r'lpips.net', help_txt) else []
print('LPIPS flag:', LPIPS or 'not available (AlexNet default)')

def check_resolution(rd, data_dir):
    # eval renders are GT | render side by side, so each half must match the input image width
    expected = imageio.imread(sorted(f for f in glob.glob(f'{data_dir}/images/**/*', recursive=True) if os.path.isfile(f))[0]).shape[1]
    actual = imageio.imread(sorted(glob.glob(f'{rd}/renders/val_step*_*.png'))[0]).shape[1] / 2
    print(f'   trained at width {actual:.0f}px (input images: {expected}px)')
    if abs(actual - expected) / expected > 0.1:
        raise RuntimeError(f'Trained at {actual:.0f}px instead of {expected}px: wrong data_factor? See F1.2')

def train(run, data_dir, factor):
    rd = f'{PROJECT}/results/{run}'
    stats = sorted(glob.glob(f'{rd}/stats/val_step*.json'))
    if stats:
        print(f'✔ {run}: already trained, loading stats'); check_resolution(rd, data_dir)
        s = json.load(open(stats[-1])); s['train_min'] = None; return s
    cmd = [sys.executable, 'simple_trainer.py', 'default', '--data_dir', data_dir, '--data_factor', str(factor),
           '--result_dir', rd, '--max_steps', str(STEPS), '--eval_steps', str(STEPS), '--save_steps', str(STEPS),
           '--disable_viewer', *LPIPS, *globals().get('TRAIN_EXTRA', [])]
    os.makedirs(rd, exist_ok=True)
    t0 = last = time.time(); buf = ''; tail = []
    p = subprocess.Popen(cmd, cwd=f'{REPO}/examples', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f'{rd}/train_log.txt', 'w') as lf:
        while (chunk := p.stdout.read(512)):
            lf.write(chunk); buf += chunk
            parts = re.split(r'[\r\n]', buf); buf = parts[-1]
            for ln in filter(str.strip, parts[:-1]):
                tail = (tail + [ln])[-40:]
                if time.time() - last > 60:
                    print(f'[{(time.time()-t0)/60:4.1f} min] {ln.strip()[:140]}'); last = time.time()
    if p.wait() != 0:
        print('\n'.join(tail)); raise RuntimeError(f'training failed, log: {rd}/train_log.txt')
    s = json.load(open(sorted(glob.glob(f'{rd}/stats/val_step*.json'))[-1]))
    s['train_min'] = round((time.time() - t0) / 60, 1)
    check_resolution(rd, data_dir)
    print(f"✔ {run}: PSNR {s['psnr']:.2f} | SSIM {s['ssim']:.3f} | LPIPS {s['lpips']:.3f} | "
          f"{s.get('num_GS', 0):,} GS | {s['train_min']} min")
    return s

RUN = f'phase1b_{SCENE}_{STEPS//1000}k'
RESULT_DIR = f'{PROJECT}/results/{RUN}'
m = train(RUN, SCENE_DIR, 1)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items()})
for f in sorted(glob.glob(f'{RESULT_DIR}/renders/val_step*_*.png'))[:3]:
    plt.figure(figsize=(16, 5)); plt.imshow(imageio.imread(f)); plt.axis('off'); plt.title(os.path.basename(f)); plt.show()

## 9. Export the `.ply` and log the run

In [ ]:
import torch, csv, datetime
from plyfile import PlyData, PlyElement

def ckpt_to_ply(ckpt_path, ply_path):
    s = torch.load(ckpt_path, map_location='cpu', weights_only=False)['splats']
    means = s['means'].numpy()
    sh0 = s['sh0'].transpose(1, 2).flatten(1).numpy()            # [N,1,3] -> [N,3]
    shN = s['shN'].transpose(1, 2).flatten(1).numpy()            # [N,K,3] -> [N,3K] (channel-major, 3DGS convention)
    opac = s['opacities'].numpy()[:, None]                       # logit space
    scales = s['scales'].numpy()                                 # log space
    quats = s['quats'].numpy()                                   # wxyz
    normals = np.zeros_like(means)
    attrs = ['x','y','z','nx','ny','nz'] + [f'f_dc_{i}' for i in range(sh0.shape[1])] \
          + [f'f_rest_{i}' for i in range(shN.shape[1])] + ['opacity'] \
          + [f'scale_{i}' for i in range(3)] + [f'rot_{i}' for i in range(4)]
    data = np.concatenate([means, normals, sh0, shN, opac, scales, quats], axis=1).astype(np.float32)
    el = np.empty(data.shape[0], dtype=[(a, 'f4') for a in attrs])
    el[:] = list(map(tuple, data))
    PlyData([PlyElement.describe(el, 'vertex')]).write(ply_path)
    return data.shape[0]

ckpts = sorted(glob.glob(f'{RESULT_DIR}/ckpts/*.pt'))
assert ckpts, 'No checkpoint found'
PLY_PATH = f'{PROJECT}/exports/{RUN}.ply'
n = ckpt_to_ply(ckpts[-1], PLY_PATH)
print(f'{n:,} Gaussians → {PLY_PATH} ({os.path.getsize(PLY_PATH)/1e6:.0f} MB)')

CSV = f'{PROJECT}/results/metrics.csv'
header = next(csv.reader(open(CSV))) if os.path.exists(CSV) else \
    ['date', 'run', 'scene', 'res', 'strategy', 'steps', 'num_gaussians', 'psnr', 'ssim', 'lpips', 'gpu', 'mem_gb', 'ply_mb']
rows = [r for r in csv.DictReader(open(CSV)) if r['run'] != RUN] if os.path.exists(CSV) else []
h, w = imageio.imread(glob.glob(f'{SCENE_DIR}/images/*')[0]).shape[:2]
rows.append({'date': str(datetime.date.today()), 'run': RUN, 'scene': SCENE, 'res': f'{w}x{h}', 'strategy': 'default',
             'steps': STEPS, 'num_gaussians': n, 'psnr': round(m['psnr'], 3), 'ssim': round(m['ssim'], 4),
             'lpips': f"{round(m['lpips'], 4)} ({'vgg' if LPIPS else 'alex'})", 'gpu': torch.cuda.get_device_name(0),
             'mem_gb': m.get('mem', ''), 'ply_mb': round(os.path.getsize(PLY_PATH) / 1e6, 1)})
with open(CSV, 'w', newline='') as f:
    w_ = csv.DictWriter(f, fieldnames=header, extrasaction='ignore'); w_.writeheader(); w_.writerows(rows)
print('logged to', CSV)

from google.colab import files
files.download(PLY_PATH)

## ✅ Phase 1b checklist
- [ ] All pose QA checks pass, or the failures are understood
- [ ] The point cloud looks like your scene from above and from the side
- [ ] The `.ply` opens in [SuperSplat](https://superspl.at/editor) and looks right from **where you actually walked**
- [ ] Anything that broke → `docs/failures.md`

**Paste back into the chat:** the output of sections 4 and 6 (including the plots if you can), the metrics from section 8, and a SuperSplat screenshot from inside the scene.

**Next:** a WayveScenes101 driving scene, with moving cars, changing exposure, several cameras, and SAM 2 masks.